# Fase 1 - Modelo Predictivo NYC Taxi Trip Duration

En este notebook se entrena un modelo de regresión para predecir la duración de viajes de taxi en Nueva York usando un flujo simple de Machine Learning.

In [10]:
import sys
!{sys.executable} -m pip install scikit-learn joblib -q


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1) Importar librerías

Este bloque importa las librerías necesarias para cargar datos, procesarlos, entrenar el modelo, evaluarlo y guardarlo.

In [11]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
import joblib
from pathlib import Path

## 2) Cargar el dataset y preprocesar datos

Aquí se carga `../data/train.csv`, se seleccionan columnas clave (incluyendo fecha/hora de recogida), se eliminan nulos y se aplican filtros simples de outliers para reducir ruido en el entrenamiento.

In [12]:
columns = [
    "passenger_count",
    "pickup_longitude",
    "pickup_latitude",
    "dropoff_longitude",
    "dropoff_latitude",
    "pickup_datetime",
    "trip_duration",
]

df = pd.read_csv("../data/train.csv", usecols=columns)
df = df.dropna().copy()

# Convertir fecha/hora y eliminar posibles errores de parseo
df["pickup_datetime"] = pd.to_datetime(df["pickup_datetime"], errors="coerce")
df = df.dropna().copy()

# Filtros simples de outliers y valores no realistas
nyc_box = (
    (df["pickup_longitude"].between(-74.3, -73.6))
    & (df["dropoff_longitude"].between(-74.3, -73.6))
    & (df["pickup_latitude"].between(40.5, 41.0))
    & (df["dropoff_latitude"].between(40.5, 41.0))
)

valid_passengers = df["passenger_count"].between(1, 6)
valid_duration = df["trip_duration"].between(60, 3600)

df = df[nyc_box & valid_passengers & valid_duration].copy()

print(f"Filas después de limpieza y filtros: {len(df)}")
df.head()

Filas después de limpieza y filtros: 1437313


,pickup_datetime,passenger_count,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,trip_duration
0,2016-03-14 17:24:55,1,-73.982155,40.767937,-73.964630,40.765602,455
1,2016-06-12 00:43:35,1,-73.980415,40.738564,-73.999481,40.731152,663
2,2016-01-19 11:35:24,1,-73.979027,40.763939,-74.005333,40.710087,2124
3,2016-04-06 19:32:31,1,-74.010040,40.719971,-74.012268,40.706718,429
4,2016-03-26 13:30:55,1,-73.973053,40.793209,-73.972923,40.782520,435


## 3) Crear features de distancia, geografía y tiempo

En este bloque se calcula `trip_distance` con Haversine y se agregan variables simples que suelen mejorar el RMSE: derivadas geográficas y variables temporales.

In [13]:
def haversine_distance(lat1, lon1, lat2, lon2):
    """Calcula la distancia Haversine en kilómetros entre dos puntos geográficos."""
    r = 6371.0  # radio aproximado de la Tierra en km

    lat1_rad = np.radians(lat1)
    lon1_rad = np.radians(lon1)
    lat2_rad = np.radians(lat2)
    lon2_rad = np.radians(lon2)

    dlat = lat2_rad - lat1_rad
    dlon = lon2_rad - lon1_rad

    a = np.sin(dlat / 2) ** 2 + np.cos(lat1_rad) * np.cos(lat2_rad) * np.sin(dlon / 2) ** 2
    c = 2 * np.arcsin(np.sqrt(a))

    return r * c


# Feature principal de distancia
df["trip_distance"] = haversine_distance(
    df["pickup_latitude"],
    df["pickup_longitude"],
    df["dropoff_latitude"],
    df["dropoff_longitude"],
)

# Derivadas geográficas simples
df["delta_lat"] = df["dropoff_latitude"] - df["pickup_latitude"]
df["delta_lon"] = df["dropoff_longitude"] - df["pickup_longitude"]
df["trip_distance_sq"] = df["trip_distance"] ** 2

# Features temporales simples
df["hour"] = df["pickup_datetime"].dt.hour
df["day_of_week"] = df["pickup_datetime"].dt.dayofweek
df["is_weekend"] = (df["day_of_week"] >= 5).astype(int)

df[["trip_distance", "hour", "day_of_week", "trip_duration"]].head()

,trip_distance,hour,day_of_week,trip_duration
0,1.498521,17,0,455
1,1.805507,0,6,663
2,6.385098,11,1,2124
3,1.485498,19,2,429
4,1.188588,13,5,435


## 4) Definir `X` e `y` y dividir datos (80/20)

Se separa la variable objetivo `trip_duration`, se incluyen las nuevas features y se divide en entrenamiento/prueba.

In [14]:
feature_cols = [
    "passenger_count",
    "pickup_longitude",
    "pickup_latitude",
    "dropoff_longitude",
    "dropoff_latitude",
    "trip_distance",
    "delta_lat",
    "delta_lon",
    "trip_distance_sq",
    "hour",
    "day_of_week",
    "is_weekend",
]

X = df[feature_cols]
y = df["trip_duration"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Tamaño de entrenamiento: {X_train.shape}")
print(f"Tamaño de prueba: {X_test.shape}")

Tamaño de entrenamiento: (1149850, 12)
Tamaño de prueba: (287463, 12)


## 5) Entrenar el modelo RandomForestRegressor

Se entrena un RandomForest con ajuste moderado de hiperparámetros para mejorar generalización y RMSE sin usar técnicas complejas.

In [15]:
model = RandomForestRegressor(
    n_estimators=200,
    max_depth=20,
    min_samples_leaf=3,
    random_state=42,
    n_jobs=-1,
)
model.fit(X_train, y_train)

print("Modelo entrenado correctamente.")

Modelo entrenado correctamente.


## 6) Evaluar el modelo (RMSE, MAE y R²)

Se generan predicciones en prueba y se reportan métricas de error para validar si hubo mejora frente al baseline.

In [16]:
from sklearn.metrics import mean_absolute_error, r2_score

y_pred = model.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"RMSE del modelo: {rmse:,.2f}")
print(f"MAE del modelo: {mae:,.2f}")
print(f"R² del modelo: {r2:.4f}")

RMSE del modelo: 262.84
MAE del modelo: 176.32
R² del modelo: 0.7962


## 7) Guardar el modelo entrenado

Se guarda el modelo en la ruta `../models/model.pkl` para su uso posterior.

In [17]:
model_path = Path("../models/model.pkl")
model_path.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(model, model_path)

print(f"Modelo guardado en: {model_path.resolve()}")

Modelo guardado en: C:\Users\fredi\OneDrive\Desktop\Modelos1_Proyecto_sustituto\models\model.pkl


## 8) Probar el modelo con algunas filas de prueba

Se muestran predicciones del modelo junto con el valor real para una revisión rápida.

In [18]:
sample_size = 5
X_sample = X_test.head(sample_size)
y_real = y_test.head(sample_size).reset_index(drop=True)
y_sample_pred = model.predict(X_sample)

resultados = pd.DataFrame({
    "real": y_real,
    "prediccion": y_sample_pred,
})

resultados

,real,prediccion
0,641,759.017600
1,226,332.191787
2,978,869.838473
3,464,540.244090
4,464,577.583100


## 9) Conclusión

Con el flujo final implementado, se entrenó correctamente un modelo `RandomForestRegressor` para predecir `trip_duration`, incluyendo limpieza de datos, creación de variables de distancia/geografía/tiempo y evaluación en conjunto de prueba.

Las métricas reportadas en este notebook (RMSE, MAE y R²) muestran que el modelo captura una parte importante de la variabilidad de la duración del viaje y produce predicciones coherentes en ejemplos reales de prueba.

Además, el modelo queda persistido en `../models/model.pkl`, por lo que esta fase termina con un artefacto reutilizable para inferencia en etapas posteriores del proyecto.